In [27]:
import os
import pprint
import tempfile

from typing import Dict, Text
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

import tensorflow_recommenders as tfrs

데이터 전처리 과정

1. 데이터 불러오기
2. 필요한 col만 남기고 지우기 (유저아이디-영화제목/ 영화제목)
3. Train/ Test 분할
3. unique 변수 생성

In [28]:
# --- 데이터 세트 준비
ratings = tfds.load("movielens/100k-ratings", split="train")
movies = tfds.load("movielens/100k-movies", split="train")

In [29]:
for x in ratings.take(1).as_numpy_iterator():
    pprint.pprint(x)

{'bucketized_user_age': 45.0,
 'movie_genres': array([7]),
 'movie_id': b'357',
 'movie_title': b"One Flew Over the Cuckoo's Nest (1975)",
 'raw_user_age': 46.0,
 'timestamp': 879024327,
 'user_gender': True,
 'user_id': b'138',
 'user_occupation_label': 4,
 'user_occupation_text': b'doctor',
 'user_rating': 4.0,
 'user_zip_code': b'53211'}


2026-03-05 12:24:12.529105: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [30]:
for x in movies.take(1).as_numpy_iterator():
    pprint.pprint(x)

{'movie_genres': array([4]),
 'movie_id': b'1681',
 'movie_title': b'You So Crazy (1994)'}


2026-03-05 12:24:12.567732: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [31]:
# --- 영화제목, 유저아이디 빼고 지우기
ratings = ratings.map(lambda x: {
    "movie_title": x["movie_title"],
    "user_id": x["user_id"],
})
movies = movies.map(lambda x: x["movie_title"])

In [41]:
for x in ratings.take(3).as_numpy_iterator():
    pprint.pprint(x)

print(len(ratings))

{'movie_title': b"One Flew Over the Cuckoo's Nest (1975)", 'user_id': b'138'}
{'movie_title': b'Strictly Ballroom (1992)', 'user_id': b'92'}
{'movie_title': b'Very Brady Sequel, A (1996)', 'user_id': b'301'}
100000


In [39]:
for x in movies.take(3).as_numpy_iterator():
    pprint.pprint(x)
    
print(len(movies))

b'You So Crazy (1994)'
b'Love Is All There Is (1996)'
b'Fly Away Home (1996)'
1682


In [34]:
# --- Train/ Test 분할 (랭킹데이터만)
tf.random.set_seed(42)

shuffled = ratings.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

train = shuffled.take(80_000)
test = shuffled.skip(80_000).take(20_000)

In [ ]:
# unique 변수 생성 (vocabulary로 쓸 거임.)
movie_titles = movies.batch(1_000) # movies에 영화제목만 있음
user_ids = ratings.batch(1_000_000).map(lambda x: x["user_id"]) # 랭킹에 유저아이디, 그에맞는 영화 있는거 유저아이디만 맵핑

unique_movie_titles = np.unique(np.concatenate(list(movie_titles))) # 리스트타입이 아님 왜냐면 위에서 배치 설정을 했잖슴! 그럼 데이터셋이겟지(흐르는 강물)
unique_user_ids = np.unique(np.concatenate(list(user_ids))) # np.unique는 중복제거, 정렬해주는 정리도구

unique_movie_titles[:10]

array([b"'Til There Was You (1997)", b'1-900 (1994)',
       b'101 Dalmatians (1996)', b'12 Angry Men (1957)', b'187 (1997)',
       b'2 Days in the Valley (1996)',
       b'20,000 Leagues Under the Sea (1954)',
       b'2001: A Space Odyssey (1968)',
       b'3 Ninjas: High Noon At Mega Mountain (1998)',
       b'39 Steps, The (1935)'], dtype=object)

1. **movie_titles**: 데이터 셋임 [1000개 뭉치]->[1000개 뭉치]->[1000개 뭉치] (리스트가) 흐르는 중
2. **list(movie_titles)**: 리스트에 담김 [[1000개],[1000개],[1000개]] 바구니에 담김 흐르지않음. 
                        
                        근데 리스트 안에 리스트?!(이중 리스트)
3. **np.concatenate(list(movie_titles))**: [ 1000개+1000개+1000개 ] 하나의 리스트가됨.


리스트는 멈춰서 쌓인 상태

데이터셋은 흐르는 상태

In [36]:
# --- 모델 구현
embedding_dimension = 32

# 쿼리 타워
user_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(
        vocabulary=unique_user_ids, mask_token=None),
    tf.keras.layers.Embedding(len(unique_user_ids)+1, embedding_dimension)
])

# 후보 타워
movie_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(
        vocabulary=unique_movie_titles, mask_token=None),
    tf.keras.layers.Embedding(len(unique_movie_titles)+1, embedding_dimension)
])

In [ ]:
# Train에 긍정적인 (사용자, 영화)쌍이 있다.
# 모델이 얼마나 좋은지 알아내려면
# 다른 쌍들보다 긍정적인 쌍이 훨씬 점수가 높으면 된다.

# --- 채점표 (재료)
metrics = tfrs.metrics.FactorizedTopK( # 채점표 안에 후보자들을 넣는데.
    candidates=movies.batch(128).map(movie_model) # 후보자들이란: 32개로 쪼갠 영화이름 데이터들을 영화 모델에 대응시킨 것.
    # 후보자들 = 영화리스트(객관식 보기)
)

# --- 채점기 (완성)
# 얼마나 정답에 가까운지 점수 매기고 (Retrieval: 검색하고)
# Loss:벌점 줌.
task = tfrs.tasks.Retrieval(
    metrics=metrics
)

In [ ]:
# --- 전체 모델
class MovielensModel(tfrs.Model): # tfrs.Model 기본모델이 부모라고 선언
    
    def __init__(self, user_model, movie_model): 
        super().__init__() # 부모님의.기본기능들 세팅
        self.movie_model: tf.keras.Model = movie_model
        self.user_model: tf.keras.Model = user_model
        self.task: tf.keras.layers.Layer = task
        
    def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
        user_embeddings = self.user_model(features["user_id"])
        positive_movie_embeddings = self.movie_model(features["movie_title"])
        return self.task(user_embeddings, positive_movie_embeddings)

In [ ]:
# ARE YOU WINNING SON?!!!!!!!!!!!